In [0]:
dbutils.widgets.text("init_load_flag","1")
init_load_flag = int(dbutils.widgets.get("init_load_flag"))
init_load_flag

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### **Data Reading From Source**

In [0]:
df = spark.sql("select * from dbcatalog.silver.customers_silver")

In [0]:
df.display()

### **Removing Duplicates**

In [0]:
df = df.dropDuplicates(['customer_id'])
df.display()

**Dividing new v/s old records**

In [0]:
if init_load_flag == 0:
    df_old = spark.sql("select customer_id, Dim_Customer_Key, create_date,  update_date from dbcatalog.gold.dim_customers")
else:
    df_old = spark.sql("select 0 customer_id,0 Dim_Customer_Key,0 create_date, 0 update_date from dbcatalog.silver.customers_silver where 1=0")


In [0]:
df_old = df_old.withColumnRenamed("Dim_Customer_Key","Old_Dim_Customer_Key")\
              .withColumnRenamed("create_date","Old_create_date")\
              .withColumnRenamed("update_date","Old_update_date")\
              .withColumnRenamed("customer_id","Old_customer_id")
df_old.display()

**applying join with old records**

In [0]:
df_join = df.join(df_old, df.customer_id == df_old.Old_customer_id,'left')
df_join.display()

**Seperating new vs old records**

In [0]:
df_new = df_join.filter(df_join.Old_Dim_Customer_Key.isNull())

df_old = df_join.filter(df_join.Old_Dim_Customer_Key.isNotNull())

In [0]:
df_old = df_old.drop('Old_customer_id','Old_update_date')
df_old = df_old.withColumnRenamed("Old_Dim_Customer_Key","Dim_Customer_Key")

df_old = df_old.withColumnRenamed("old_create_date","create_date")
df_old = df_old.withColumn("create_date", to_timestamp(col("create_date")))

df_old = df_old.withColumn("update_date", current_timestamp())

df_old.display()

**Preparing df_new**

In [0]:
df_new = df_new.drop('Old_Dim_Customer_Key','Old_customer_id','Old_create_date','Old_update_date')

df_new = df_new.withColumn("update_date",current_timestamp())
df_new = df_new.withColumn("create_date",current_timestamp())

df_new.display()

### **Surrgate keys**

In [0]:
df_new  = df_new.withColumn("Dim_Customer_Key", monotonically_increasing_id()+lit(1))
df_new.display()

In [0]:
# Max surrogate key

if init_load_flag == 1:
    max_surrogate_key = 0
else:
    max_surrogate_key = spark.sql("select max(Dim_Customer_Key) as max_surrogate_key from dbcatalog.gold.dim_customers").collect()[0]['max_surrogate_key']

In [0]:
df_new = df_new.withColumn("Dim_Customer_Key", lit(max_surrogate_key)+ col("Dim_Customer_Key"))

**Union of df_new and df_old**

In [0]:
df_final = df_new.unionByName(df_old)

In [0]:
df_new.printSchema()
df_old.printSchema()

In [0]:
df_final.display()

In [0]:
df_old.printSchema()
df_new.printSchema()

In [0]:
from delta.tables import DeltaTable

In [0]:
if init_load_flag == 1:
  df_final.write.mode("overwrite")\
    .option("path","abfss://gold@storageaccnitin.dfs.core.windows.net/dim_customers")\
    .saveAsTable("dbcatalog.gold.dim_customers")
else:
  dlt_obj = DeltaTable.forPath(spark,'abfss://gold@storageaccnitin.dfs.core.windows.net/dim_customers')
  dlt_obj.alias("trg").merge(
    df_final.alias("src"),
    "trg.Dim_Customer_Key = src.Dim_Customer_Key")\
  .whenMatchedUpdateAll()\
  .whenNotMatchedInsertAll().execute()

In [0]:
%sql
select * from dbcatalog.gold.dim_customers;